# Project 07: Burrows-Wheeler Transform

**Note**: Keep the cell below as is. It is for formatting purposes

In [1]:
%%html
<style>
.MathJax_Display,  /* MathJax 2 */
mjx-container {    /* MathJax 3 */
  text-align: left !important;
}
</style>

## Learning Objectives

1. Students will be able to explain the fundamental principles of the Burrows-Wheeler Transform algorithm, including its purpose, process, and applications in data compression and bioinformatics.
2. Students will be able to implement the Burrows-Wheeler Transform using both the naive approach and the more efficient suffix array method.
3. Students will be able to construct auxiliary data structures (count array and occurrence array) needed for efficient pattern matching with the FM-Index.
4. Students will be able to apply the BWT-based pattern matching algorithm to find exact string matches within a reference text.

---

## Background
The Burrows-Wheeler Transform (BWT) is a powerful algorithm used in data compression and bioinformatics. Developed by Michael Burrows and David Wheeler in 1994, it rearranges a character string into runs of similar characters, making it more amenable to compression techniques like move-to-front encoding and run-length encoding. Despite adding a small amount of overhead, BWT's ability to group similar characters together significantly improves compression efficiency.

The BWT works by sorting all possible rotations of a string lexicographically and then extracting the last column of the sorted matrix. What makes BWT remarkable is that this transformation is reversible without storing much additional information - only the position of the original string in the sorted rotations is needed.

In recent years, BWT has found extensive applications beyond compression. It's particularly valuable in bioinformatics for efficient sequence alignment of DNA reads against reference genomes, as implemented in popular tools like Bowtie and BWA. The algorithm's efficiency in handling large datasets has made it indispensable for processing the massive amounts of data generated by next-generation sequencing technologies.

## The BWT Algorithm

![](assets/rotation.png)

> Credit: Ferragina, P., Giancarlo, R., Manzini, G., & Sciortino, M. (2005). Boosting textual compression in optimal linear time. In Journal of the ACM (Vol. 52, Issue 4, pp. 688–713). Association for Computing Machinery (ACM). https://doi.org/10.1145/1082036.1082043 

At its core, the BWT algorithm involves creating all cyclic rotations of an input string, sorting them lexicographically, and then extracting the last column.

Let's define the pseudocode for our BWT function:

$$
{\small
\begin{array}{l}
\textbf{function } \text{BWT}(text) \\
\quad \text{Append a unique end-of-string character to } text \text{ if not present} \\
\quad \text{Create a list to store all possible rotations of the text} \\
\quad \textbf{for each } \text{possible starting position in the text} \textbf{ do} \\
\quad\quad \text{Generate a rotation starting at this position and add to the list} \\
\quad \text{Sort all rotations lexicographically} \\
\quad \text{Initialize an empty string for the transformed result} \\
\quad \textbf{for each } \text{rotation in the sorted list} \textbf{ do} \\
\quad\quad \text{Append the last character of this rotation to the transformed result} \\
\quad \textbf{return } \text{transformed result and position of original text in sorted rotations}
\end{array}
}
$$

In [2]:
from typing import Dict, List, Tuple


def normalize_text(text: str) -> str:
    """
    Ensures consistent lexicographic behavior by converting
    string to uppercase and appending a unique sentinel character '$' if
    not already present
    Args:
        text (str): Input string to normalize
    Returns:
        Normalized string in uppercase ending with sentinel '$'
    """

    text = text.upper()  # Convert all chars to uppercase letters
    # Add sentinel if not included, skip code block if present
    if '$' not in text:
        text += '$'

    return text


def generate_rotations(text: str) -> list[str]:
    """
    For string of length n, function produces n rotations where each rotation
    shifts starting position by one character
    Args:
        text (str): Input string of length n, normalized to uppercase with sentinel '$'
    Returns:
        sorted_rotations (str): List of all cyclic rotations of input string sorted lecicographically
    """

    rotations = [text]  # Store original string as first rotation

    # Iterate over each rotation offset
    for i in range(len(text) - 1):
        # Slice and concatenate to rotate left by i+1 positions
        rotated = text[i + 1:] + text[:i + 1]
        # Append generated rotation to list
        rotations.append(rotated)

    # Sort the full list of rotations (must sort the LIST, not a single string)
    sorted_rotations = sorted(rotations)

    # Return the sorted list AFTER the loop completes
    return sorted_rotations


def extract_last_column(sorted_rotations: list[str]) -> str:
    """
    The Burrows–Wheeler Transform is defined as the sequence of last
    characters from each sorted rotation
    Parameter:
        sorted_rotations (list): Lexicographically sorted list of cyclic rotations.
    Returns:
        The Burrows–Wheeler Transform string
    """

    # Each rotation is same length, so last character is rotation[-1]
    return ''.join(rotation[-1] for rotation in sorted_rotations)


def BWT(string: str) -> str:
    """
    Compute the Burrows–Wheeler Transform (BWT) of a string
    This function orchestrates the four conceptual steps of BWT construction:
        1. Normalize input (uppercase + sentinel)
        2. Generate all cyclic rotations
        3. Sort rotations lexicographically
        4. Extract the last column of the sorted matrix
    Parameter:
        sequence(str): Input string to transform
    Returns:
        BWT (str): Burrows–Wheeler Transform of input sequence
    """

    # Apply helper function to normalize text string
    text = normalize_text(string)
    # Apply helper function to generate rotations of text
    sorted_rotations = generate_rotations(text)
    # Return BWT
    return extract_last_column(sorted_rotations)

While this approach works, it's inefficient for large strings. A more efficient implementation uses suffix arrays.

## Suffix Arrays for BWT

Suffix arrays provide a memory-efficient way to compute the BWT. A suffix array contains the starting positions of all suffixes of a string, sorted lexicographically.

$$
\small
\begin{array}{l}
\textbf{function } \text{suffix\_array}(text) \\
\quad \text{Create a list of all suffixes of the text, each paired with its starting position} \\
\quad \text{Sort these suffix-position pairs lexicographically by suffix} \\
\quad \text{Extract just the starting positions from the sorted pairs} \\
\quad \textbf{return } \text{the list of sorted starting positions}
\end{array}
$$


In [3]:
def suffix_array(string: str) -> list[int]:
    """Function to calculate suffix-array for a given string.
    
    Computes the suffix array by sorting all suffixes of the input string
    lexicographically and returning their starting positions.
    
    Args:
        string: The input string to process.
    
    Returns:
        A list of integers representing the starting positions of the
        lexicographically sorted suffixes.
        
    Examples:
        >>> suffix_array('googol')
        [6, 3, 0, 5, 2, 4, 1]
        
        >>> suffix_array('banana$')
        [6, 5, 3, 1, 0, 4, 2]
    """
    # append a unique end-of-string char ($) to text if not present
    if '$' not in string:
        string += '$'
    # Create an list to store all possible rotations
    suffix_rotations = []

   #for each possible starting position
    for i in range(len(string)):
        # Get the suffix of the rotated string
        rotated_string_suffix = string[i:]

        # Add to the list paired with its starting position
        suffix_rotations.append([i, rotated_string_suffix])

    # Sort the pairs by suffix
    sorted_rotations = sorted(suffix_rotations, key=lambda x: x[1])

    # Extract just the starting positions
    sorted_positions = []
    sorted_positions.extend(sorted_rotations[i][0] for i in range(len(sorted_rotations)))

    # Return the list of sorted starting positions
    return sorted_positions

Once we have the suffix array, we can compute the BWT more efficiently:

$$
\small
\begin{array}{l}
\textbf{function } \text{BWT\_from\_suffix\_array}(text, \text{suffix\_positions}) \\
\quad \text{Initialize an empty string of the same length as the input text} \\
\quad \textbf{for each } \text{position in the suffix positions list do} \\
\quad\quad \textbf{if } \text{the suffix starts at the beginning of the text then} \\
\quad\quad\quad \text{Add the last character of the text to the transformed result} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character preceding the suffix to the transformed result} \\
\quad \textbf{return } \text{the transformed result}
\end{array}
$$

![](assets/full.png)

> Credit: Siren, J. (2016). Burrows-Wheeler Transform for Terabases. In 2016 Data Compression Conference (DCC) (pp. 211–220). 2016 Data Compression Conference (DCC). IEEE. https://doi.org/10.1109/dcc.2016.17 

In [4]:
def BWT_from_suffix_array(text: str, suffix_positions: list[int]) -> str:
    """Compute the Burrows–Wheeler Transform from a suffix array
    Parameters:
        text: The input string (must include a unique end marker like '$')
        suffix_positions: The suffix array giving starting indices of all
            suffixes in lexicographic order
    Returns:
        The Burrows–Wheeler Transform string
    """
    n = len(text)  # store length of the input string
    bwt_chars = []  # list to accumulate BWT characters

    for pos in suffix_positions:  # iterate through suffix array positions in sorted order

        if pos == 0:  # if the suffix starts at index 0
            bwt_chars.append(text[-1])  # append the last character of the string
        else:
            bwt_chars.append(text[pos - 1])  # otherwise append the character preceding the suffix

    return "".join(bwt_chars)  # join list into final BWT string

---

## FM-Index for Pattern Matching

The BWT becomes particularly powerful when combined with auxiliary data structures to form an FM-Index, which enables efficient pattern matching. Two key components are the count and occurrence arrays.

The FM-index is a compressed full-text index data structure that combines the Burrows-Wheeler Transform (BWT) with auxiliary data structures to enable efficient pattern matching while maintaining a small memory footprint. Developed by Paolo Ferragina and Giovanni Manzini (hence the name "FM"), it's sometimes referred to as "Full-text Minute-space" index.

#### Core Components

The FM-index consists of several key components:

1. **The BWT (L column)** - The last column from the Burrows-Wheeler Matrix, which is a permutation of the original text that groups similar characters together

2. **First column (F)** - Can be represented as an array of |Σ| integers (where Σ is the alphabet), or sometimes not stored at all as it can be derived

3. **Count array (C)** - Stores the number of characters lexicographically smaller than each character in the alphabet

4. **Occurrence table (OT)** - Tracks how many times each character appears up to each position in the BWT

5. **Suffix Array (SA) sample** - A subset of the suffix array positions, often stored at regular intervals to save space

#### Key Operations

The FM-index supports several fundamental operations:

1. **Count** - Returns the number of occurrences of a pattern in the text in O(p) time, where p is the pattern length

2. **Locate** - Returns the positions of all occurrences in the text

3. **Backward search** - The core algorithm that enables efficient pattern matching by traversing the pattern from right to left

4. **LF mapping** - A procedure that maps positions in the L column to corresponding positions in the F column

#### Advantages

The FM-index offers several notable benefits:

1. **Space efficiency** - Occupies space close to the entropy of the indexed text, typically 5nHk(T) + o(n) bits, where Hk(T) is the k-th order entropy

2. **Query performance** - Allows searching for pattern occurrences in O(p + occ log^(1+ε) n) time, where occ is the number of occurrences

3. **Self-indexing** - Encapsulates the indexed data, allowing reconstruction of the original text without storing it separately

4. **Compression** - Takes advantage of the compressibility of the indexed data

5. **Partial decompression** - Only decompresses tiny portions of the compressed file during queries

The FM-index has become particularly important in bioinformatics for efficient DNA and protein sequence alignment, forming the basis of popular tools like Bowtie and BWA that handle massive genomic datasets.

### The implementation

One of the key parts for string matching is to do Last-to-First column mapping (LF mapping) within the BWT matrix. With the LF property, we  need to build two dictionaries for our reference string beforehand:

1. count: e.g.  `{'A': 0, 'C': 2, 'G': 3, 'T': 5}`

Where for each character `a` in a string, `count[a]` contains the number of characters in string that are lexicographically smaller than `a`.

2. occur: `{'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}`

Where for each character `a` in a bwt string, `occur[a][i]` contains the number of occurences of `a` in `bwt_string[0,i], i=1,...,len(bwt_string)` (i.e. the first i characters in bwt string).

With those two dictionaries, we can then start matching the query string to our reference string. 

## Count Array

The count array stores the number of characters lexicographically smaller than each character in the alphabet:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_counts}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary to track character counts} \\
\quad \text{Initialize a result dictionary for cumulative counts} \\
\quad \text{Set initial cumulative count to zero} \\
\quad \textbf{for each } \text{character in the sorted alphabet do} \\
\quad\quad \text{Store the current cumulative count for this character} \\
\quad\quad \text{Increase the cumulative count by the frequency of this character} \\
\quad \textbf{return } \text{the dictionary of cumulative counts}
\end{array}
$$

In [5]:
from collections import Counter

def cal_count(string: str) -> Dict[str, int]:
    """
    For each character in alphabet of input string, calculates how many characters
    in string are lexicographically smaller. Standard C-array used in FM-index backward search
    Parameter:
        BWT (str): input used to compute cumulative counts
    Returns:
        count_array (dict): maps each character to number of chars in string  lexicographically
        smaller than that char
    """

    # Count occurrences of each char
    char_counts = Counter(string)

    # Sort characters lexicographically
    sorted_char_counts = dict(sorted(char_counts.items(), key=lambda x: x[0]))

    count_array: dict[str, int] = {}
    cumulative_count = 0

    # For each character in sorted order, store cumulative count of smaller chars
    for char in sorted_char_counts.keys():
        count_array[char] = cumulative_count
        cumulative_count += char_counts[char]

    return count_array


## Occurrence Array

The occurrence array tracks how many times each character appears up to each position in the BWT:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_occurrences}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary mapping each character to an array of zeros} \\
\quad \textbf{for each } \text{position in the transformed text do} \\
\quad\quad \text{Identify the character at the current position} \\
\quad\quad \textbf{for each } \text{character in the alphabet do} \\
\quad\quad\quad \text{Copy the previous occurrence count to the current position} \\
\quad\quad \text{Increment the occurrence count for the current character} \\
\quad \textbf{return } \text{the dictionary of occurrence counts}
\end{array}
$$


In [6]:
def cal_occur(bwt_string: str) -> Dict[str, List[int]]:
    """
    Builds occurrence table used in FM-index backward search. For
    each char in alphabet of BWT string, returns a list where
    the i-th entry is number of times char appears in BWT
    up to and including position i.
    Parameter::
        bwt_string (str): Burrows–Wheeler transformed string with sentinel char
    Returns:
        occur (dict): dictionary mapping each char to list of integers. For given
        char c, occur[c][i] is number of occurrences of c in bwt_string[0:i+1].
    """

    # Extract alphabet from BWT string (unique characters only)
    alphabet = sorted(set(bwt_string))

    # Initialize occurrence table: each char maps to list of zeros
    occur: Dict[str, List[int]] = {
        char: [0] * len(bwt_string) for char in alphabet
    }

    # Iterate through each position in BWT string
    for i, char in enumerate(bwt_string):
        # if index greater than 0 (not first char in string)
        if i > 0:
            # Copy previous counts into current position for all characters
            for c in alphabet:
                occur[c][i] = occur[c][i - 1]

        # Increment count for character at current position
        occur[char][i] += 1

    return occur


---

# Pattern Matching with BWT

![](assets/pattern_match.png)

> Credit: ["DNA Sequence Alignment with the Burrows Wheeler Transform"](https://cs.carleton.edu/cs_comps/2324/sequenceAlignment/) from Carleton College

## Alignments using BWT

With the BWT and auxiliary structures in place, we can efficiently find pattern matches using backward search:

$$
\small
\begin{array}{l}
\textbf{function } \text{update\_range}(\text{character}, \text{range\_start}, \text{range\_end}, \text{counts}, \text{occurrences}) \\
\quad \text{Calculate new start position using the character's count and occurrences at range start} \\
\quad \text{Calculate new end position using the character's count and occurrences at range end} \\
\quad \textbf{return } \text{new start and end positions}
\end{array}
$$

In [7]:
def update_range(
    lower: int,
    upper: int,
    count: Dict[str, int],
    occur: Dict[str, List[int]],
    a: str
) -> Tuple[int, int]:
    """
    During backward search in the FM-index, updates current
    [lower, upper] range in suffix array corresponding to pattern
    suffix processed so far, when a new character a is prepended.
    Notes: This function assumes that the character a is present in the count
           dictionary and that occur[a] is defined for all positions in the BWT.
    The update uses the standard FM-index formulas:
        lower = C[a] + Occ(a, lower - 1)
        upper = C[a] + Occ(a, upper) - 1
    with convention that Occ(a, -1) = 0, which is handled by a special
    case when lower == 0.
    Parameters:
        lower:
            Current lower boundary (inclusive) of suffix array range.
        upper:
            Current upper boundary (inclusive) of the suffix array range.
        count:
            C-array mapping each character to the number of characters
            lexicographically smaller than it in the BWT.
        occur:
            Occurrence table mapping each character to a list of cumulative
            occurrence counts at each position in the BWT.
        a:
            The character being processed in the pattern (moving right to left).
    Returns:
        lower_new, upper_new (tuple): updated range
    """
    # Handle Occurrence (a, lower - 1) with boundary condition at lower == 0
    if lower == 0:
        lower_new = count[a]
    else:
        lower_new = count[a] + occur[a][lower - 1]

    # Upper bound uses Occ(a, upper) directly
    upper_new = count[a] + occur[a][upper] - 1

    return lower_new, upper_new


<div style="text-align: left">
    
$$
\small
\begin{array}{l}
\textbf{function } \text{find\_match}(\text{pattern}, \text{transformed}, \text{counts}, \text{occurrences}, \text{suffix\_positions}) \\
\quad \text{Initialize search range to cover the entire transformed text} \\
\quad \textbf{for each } \text{character in the pattern, processing from right to left do} \\
\quad\quad \text{Update the search range based on the current character} \\
\quad\quad \textbf{if } \text{the range becomes empty then} \\
\quad\quad\quad \textbf{return } \text{empty list as pattern is not found} \\
\quad \text{Collect all suffix positions within the final range} \\
\quad \textbf{return } \text{the list of matching positions}
\end{array}
$$

</div>

In [8]:
def find_match(query: str, reference: str) -> List[int]:
    """
    Builds an FM-index implicitly from reference string by computing
    its suffix array and BWT, then performs backward search to find
    all occurrences of the query. It returns the starting positions (0-based)
    of all exact matches of the query in the reference.
    Parameters::
        query (str): Pattern string to search for
        reference (str): Text string to search within with sentinel char at end
    Returns:
        A sorted list of 0-based starting positions of all occurrences of the
        query in the reference. If no matches are found, an empty list is
        returned.
    """

    # Ensure reference has a sentinel marker
    if '$' not in reference:
        reference = reference + '$'

    # Build suffix array and BWT
    suffix_pos = suffix_array(reference)
    bwt = BWT_from_suffix_array(reference, suffix_pos)

    # Initialize search range over the entire BWT
    lower = 0
    upper = len(bwt) - 1

    # Precompute C-array and Occ table
    count = cal_count(bwt)
    occur = cal_occur(bwt)

    # Process query characters from right to left (backward search)
    for char in reversed(query):
        # If character not in alphabet, no matches are possible
        if char not in count:
            return []

        # Update the search range
        lower, upper = update_range(lower, upper, count, occur, char)

        # If the range becomes empty, there are no matches
        if lower > upper:
            return []

    # Collect all matching positions in the final range
    return sorted(suffix_pos[lower: upper + 1])

***

## Run-Length Encoding

The Burrows-Wheeler Transform tends to group identical characters into runs, which makes it a natural candidate for run-length encoding (RLE). In RLE, we store each run as a pair: the character and the length of its consecutive occurrence. This is particularly effective when the transformed text contains many long runs.

Conceptually, run-length encoding on the BWT string works as follows:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_encode}(\text{input text}) \\
\quad \text{Start with an empty encoded text} \\
\quad \textbf{if } \text{the input text is empty then} \\
\quad\quad \textbf{return } \text{the empty encoded text} \\
\quad \text{Remember the first character of the input text} \\
\quad \text{Set a counter to one for how many times this character appears in a row} \\
\quad \textbf{for each } \text{next character in the input text} \textbf{ do} \\
\quad\quad \textbf{if } \text{the character is the same as the one being counted} \\
\quad\quad\quad \text{Increase the counter by one} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character being counted and its counter to the encoded text} \\
\quad\quad\quad \text{Begin counting from one for the new character} \\
\quad \text{After the scan, add the final character and its counter to the encoded text} \\
\quad \textbf{return } \text{the encoded text}
\end{array}
$$


In [9]:
def run_length_encode(bwt_string: str) -> str:
    """
    Scans the input string from left to right, groups consecutive
    identical characters into runs, and returns an encoded string where each
    run is represented as the character followed by its count (in decimal).
    Parameter:
        bwt_string (str): Input string to encode, typically a BWT string but not
            restricted to that
    Returns:
        encoded_string(str): run-length encoded string of form 'a3n2b1$1a2', where each
        character is followed by length of its consecutive run.
    """

    # Handle empty input
    if not bwt_string:
        return ""

    encoded_string = ""

    # Initialize current run
    symbol = bwt_string[0]
    counter = 1

    # Iterate through the string starting from the second character
    for i in range(1, len(bwt_string)):
        if bwt_string[i] == symbol:
            # Extend current run
            counter += 1

        else:
            # Close current run and start a new one
            encoded_string += symbol + str(counter)
            symbol = bwt_string[i]
            counter = 1

    # Append the final run
    encoded_string += symbol + str(counter)

    return encoded_string

***

## Run-Length Decoding

Run-length decoding reverses the RLE process by expanding each stored pair back into its original sequence of repeated characters. When combined with the inverse BWT, this allows us to reconstruct the original text from a compressed, run-length encoded representation of the transform.

The pseudocode for decoding is straightforward:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_decode}(\text{encoded text}) \\
\quad \text{Start with an empty decoded text} \\
\quad \text{Begin reading at the start of the encoded text} \\
\quad \textbf{while } \text{there is still unread encoded text} \textbf{ do} \\
\quad\quad \text{Read a single character, which is the symbol to be repeated} \\
\quad\quad \text{Then read the following one or more digits as the number of repeats} \\
\quad\quad \text{Turn this sequence of digits into a number} \\
\quad\quad \text{Add that many copies of the symbol to the decoded text} \\
\quad \textbf{return } \text{the decoded text}
\end{array}
$$


In [10]:
def run_length_decode(encoded: str) -> str:
    """
    Reconstructs original string by parsing an encoded
    representation where each run is stored as a character followed by its
    count (in decimal), and expanding each run back into repeated characters.
    Note:
        This implementation assumes that the encoding format is strictly
        alternating single characters and their counts, with counts written
        as one or more decimal digits (e.g., 'a12b3'). It parses digits
        following each character until a non-digit is encountered.
    Parameter::
        encoded (str): A run-length encoded string of the form 'a3n2b1$1a2', or more
            generally 'char + integer' repeated.
    Returns:
        decoded_string (str): The decoded string obtained by expanding all runs in the encoded input.
    """
    decoded_string = ""
    i = 0
    n = len(encoded)

    while i < n:
        # Current symbol
        symbol = encoded[i]
        i += 1

        # Parse one or more digits for the count
        count_str = ""
        while i < n and encoded[i].isdigit():
            count_str += encoded[i]
            i += 1

        # Convert count and expand run
        count = int(count_str)
        decoded_string += symbol * count

    return decoded_string


In [11]:
# Inverse BWT
def inverse_BWT(bwt: str) -> str:
    """
    Reconstruct original string from its Burrows–Wheeler Transform.
    Standard LF-mapping reconstruction.

    Parameter:
        bwt (str): BWT string containing exactly one sentinel '$'
    Returns:
        original (str): Original string reconstructed from BWT
    """

    # Initialize table of empty strings
    table = [""] * len(bwt)

    # Repeat len(bwt) times to reconstruct full matrix
    for _ in range(len(bwt)):
        # Prepend BWT characters to each row
        table = sorted([bwt[i] + table[i] for i in range(len(bwt))])

    # Return the row ending with sentinel
    for row in table:
        if row.endswith("$"):
            return row


In [12]:
# Implementation Block
if __name__ == "__main__":
    # Test sequence
    seq = "AATACGACTTGAAGTAGA"

    print("Original sequence:")
    print(seq)

    # Compute BWT
    bwt_seq = BWT(seq)
    print("\nBWT of sequence:")
    print(bwt_seq)

    # Build suffix array and BWT
    suffix_pos = suffix_array(seq+'$')
    bwt_suffix = BWT_from_suffix_array(seq+'$', suffix_pos)
    print("\nBWT of sequence (built from suffix array -- quicker):")
    print(bwt_suffix)

    # Inverse BWT
    original_from_bwt = inverse_BWT(bwt_seq)
    print("\nReconstructed original from BWT:")
    print(original_from_bwt)

    # Run-length encode BWT
    rle = run_length_encode(bwt_seq)
    print("\nRun-length encoded BWT:")
    print(rle)

    # Decode RLE
    decoded_rle = run_length_decode(rle)
    print("\nDecoded run-length encoding:")
    print(decoded_rle)

    # Pattern matching with FM-index
    query = "CTTG"
    position = find_match(query, seq)
    print("\nPattern matching with FM-index:")
    print(f"Sequence found at starting position(s) {position}")


Original sequence:
AATACGACTTGAAGTAGA

BWT of sequence:
AGG$TGTAAAAATCAAGTC

BWT of sequence (built from suffix array -- quicker):
AGG$TGTAAAAATCAAGTC

Reconstructed original from BWT:
AATACGACTTGAAGTAGA$

Run-length encoded BWT:
A1G2$1T1G1T1A5T1C1A2G1T1C1

Decoded run-length encoding:
AGG$TGTAAAAATCAAGTC

Pattern matching with FM-index:
Sequence found at starting position(s) [7]


In [13]:
# Output from executing code with implementation string:
# Original sequence:
# AATACGACTTGAAGTAGA

# BWT of sequence:
# AGG$TGTAAAAATCAAGTC

# BWT of sequence (built from suffix array -- quicker):
# AGG$TGTAAAAATCAAGTC

# Reconstructed original from BWT:
# AATACGACTTGAAGTAGA$

# Run-length encoded BWT:
# A1G2$1T1G1T1A5T1C1A2G1T1C1

# Decoded run-length encoding:
# AGG$TGTAAAAATCAAGTC

# Pattern matching with FM-index:
# Sequence found at starting position(s): [7]

## Tricky Parts to Watch Out For

1. **End-of-string character**: Ensure your implementation includes a unique end-of-string character (often '$') that doesn't appear elsewhere in the string. This character should typically be lexicographically smaller than all other characters.

2. **Indexing issues**: Pay careful attention to zero-based versus one-based indexing, especially when implementing the BWT from suffix arrays.

3. **Edge cases**: Handle edge cases properly, such as when the suffix array index is 0, or when dealing with empty strings or patterns.

4. **Performance considerations**: The naive BWT implementation is $O(n^2 \log n)$ due to sorting n strings of length n. Using suffix arrays reduces this to $O(n \log n)$.

5. **Memory usage**: For large strings, storing all rotations can be memory-intensive. The suffix array approach is more memory-efficient.

---

# Summary

The Burrows-Wheeler Transform showcases how a seemingly simple reordering of characters can lead to powerful applications in compression and pattern matching. By understanding both the conceptual foundation and the implementation details, you'll gain insight into one of the most elegant algorithms in computer science, with applications ranging from everyday file compression to cutting-edge genomic research.